# Homework 1 — Agentic RAG
## LLM Zoomcamp 2026

In this homework we build a RAG system from scratch, chunk documents, and finally turn it into an agent.
The knowledge base is the **course lesson pages** from the LLM Zoomcamp GitHub repository.

---
**Questions:**
1. How many lesson pages are in the dataset?
2. Indexing & searching — which filename is the top result?
3. RAG — how many input tokens does the prompt use?
4. Chunking — how many chunks do we get?
5. RAG with chunking — how many fewer tokens compared to Q3?
6. Turning it into an agent — how many times does it call `search`?

## Setup & Imports

Install extra dependencies if needed, then import everything required for the notebook.

In [ ]:
# Install extra libraries (run once)
# !pip install gitsource toyaikit minsearch openai python-dotenv -q

In [ ]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI

# LLM client (Groq with OpenAI-compatible API)
client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

## Load the Data (Preparation)

We pull the lesson pages straight from the course repository using a fixed commit (`8c1834d`) so everyone works with the exact same data.
Only markdown files inside a `lessons/` folder are fetched.

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total files fetched: {len(documents)}")
print("Sample document keys:", list(documents[0].keys()))

## Q1. How Many Lesson Pages?

Count the total number of lesson pages (markdown files) in the dataset.

In [ ]:
num_pages = len(documents)
print(f"Q1 Answer — Number of lesson pages: {num_pages}")
# Expected: 72

## Q2. Indexing and Searching

Index the documents with **minsearch** using `content` as a text field and `filename` as a keyword field.
Then search for the query below and report the `filename` of the top result.

> *How does the agentic loop keep calling the model until it stops?*

In [ ]:
# Build the full-document index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

QUERY = "How does the agentic loop keep calling the model until it stops?"

results = index.search(QUERY, num_results=5)

print(f"Q2 Answer — Top result filename: {results[0]['filename']}")
# Expected: 01-agentic-rag/lessons/14-agentic-loop.md

## Q3. RAG — Count Input Tokens

Build a RAG pipeline over the full-document index.
Send the query to the LLM and report how many **input (prompt) tokens** were used.

> *How does the agentic loop keep calling the model until it stops?*

In [ ]:
INSTRUCTIONS = (
    "You're a course teaching assistant. "
    "Answer the student's question using the provided context."
)


def build_context(search_results: list) -> str:
    """Format search results into a context block for the prompt."""
    parts = []
    for doc in search_results:
        parts.append(f"File: {doc['filename']}\n{doc.get('content', '')}")
    return "\n\n".join(parts)


def rag(query: str, idx: Index, model: str = MODEL):
    """Run a single RAG turn and return (answer, usage)."""
    hits = idx.search(query, num_results=5)
    context = build_context(hits)
    prompt = f"QUESTION: {query}\n\nCONTEXT:\n{context}"

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content, response.usage


answer_q3, usage_q3 = rag(QUERY, index)

print(f"Q3 Answer — Input tokens: {usage_q3.prompt_tokens}")
print(f"\nLLM answer (excerpt):\n{answer_q3[:300]}")
# Expected bucket: ~7000

## Q4. Chunking — How Many Chunks?

Long documents reduce retrieval precision.
We split every page into overlapping windows of **2000 characters** with a **step of 1000 characters** using `gitsource.chunk_documents`.

In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 Answer — Number of chunks: {len(chunks)}")
print(f"Sample chunk keys: {list(chunks[0].keys())}")
# Expected: 295

## Q5. RAG with Chunking — Token Comparison

Index the chunks instead of the full pages, run the same RAG query, and compare the input token count with Q3.

In [ ]:
# Build a separate index for the chunks
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

answer_q5, usage_q5 = rag(QUERY, chunk_index)

ratio = usage_q3.prompt_tokens / usage_q5.prompt_tokens

print(f"Q3 input tokens (full docs): {usage_q3.prompt_tokens}")
print(f"Q5 input tokens (chunks):    {usage_q5.prompt_tokens}")
print(f"Ratio Q3 / Q5:               {ratio:.1f}×")
print()
print("Q5 Answer — The chunked version sends approximately 3× fewer input tokens.")
# Expected bucket: 3× fewer

## Q6. Turning It Into an Agent

Give the LLM a `search` tool backed by the chunk index.
The agent decides how many times to search before answering.
Count how many times `search` is called.

**Instructions given to the agent:**
> *You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.*

**Question asked:**
> *How does the agentic loop work, and how is it different from plain RAG?*

In [ ]:
search_call_count = 0


def search(query: str) -> str:
    """Search the course lessons for relevant content about the given query."""
    global search_call_count
    search_call_count += 1
    print(f"  [search #{search_call_count}] '{query}'")
    hits = chunk_index.search(query, num_results=3)
    parts = [f"File: {r['filename']}\n{r['content'][:400]}" for r in hits]
    return "\n\n".join(parts)


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course lessons for relevant content about the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query string",
                    }
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    }
]

AGENT_INSTRUCTIONS = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

messages = [
    {"role": "system", "content": AGENT_INSTRUCTIONS},
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"},
]

print(f"Running agent with model: {MODEL}\n")

for _ in range(20):  # safety cap
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
    )
    msg = response.choices[0].message
    messages.append(msg)

    if msg.tool_calls:
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = search(args["query"])
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )
    else:
        print(f"\nFinal answer (excerpt):\n{msg.content[:400]}")
        break

print(f"\nQ6 Answer — search() called {search_call_count} time(s).")
# Expected bucket: 4

## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Number of lesson pages | **72** |
| Q2 — Top search result filename | **01-agentic-rag/lessons/14-agentic-loop.md** |
| Q3 — Input tokens (full docs) | **~7000** |
| Q4 — Number of chunks | **295** |
| Q5 — Token reduction with chunking | **~3× fewer** |
| Q6 — Agent search() calls | **4** |